In [1]:
from tests.synthetic_data import make_sceptre_style_synth
from src.sceptre import prepare_crt_inputs
import scanpy as sc
import muon as mu
import pandas as pd
import numpy as np
import os

from src.sceptre import (
    build_ntc_group_inputs,
    compute_guide_set_null_pvals,
    crt_pvals_for_ntc_groups_ensemble,
    crt_pvals_for_ntc_groups_ensemble_skew,
    make_ntc_groups_ensemble,
    run_all_genes_union_crt,
)
from src.visualization import qq_plot_ntc_pvals


/oak/stanford/groups/engreitz/Users/ymo/miniforge3/envs/programDE/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/oak/stanford/groups/engreitz/Users/ymo/miniforge3/envs/programDE/lib/python3.14/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


In [2]:
mdata = mu.read("/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Result/020326_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50/adata/cNMF_50_0_2.h5mu")
mdata_guide = mu.read("/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Data/withguide.h5ad")

/oak/stanford/groups/engreitz/Users/ymo/miniforge3/envs/programDE/lib/python3.14/site-packages/mudata/_core/mudata.py:1598: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/oak/stanford/groups/engreitz/Users/ymo/miniforge3/envs/programDE/lib/python3.14/site-packages/mudata/_core/mudata.py:1461: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


In [ ]:
adata = mdata["cNMF"].copy()
adata.obsm["cnmf_usage"] = np.asarray(adata.X)  # ensure dense float

# Align gene modality to adata cells
g = mdata_guide[adata.obs_names].copy()

# Guide assignment (cell x guide)
adata.obsm["guide_assignment"] = g.obsm["guide_assignment"].copy()

# Program names
adata.uns["program_names"] = list(adata.var_names)

# Guide names must match columns of guide_assignment
guide_names = list(mdata_guide.uns["guide_names"])
adata.uns["guide_names"] = guide_names

# guide2gene must map guide_name -> gene name (keys must match guide_names)
'''guide2gene = (
    mdata_guide.uns["guide_targets"]
    .loc[guide_names]
    .to_dict()
)'''

guide2gene = dict(zip(guide_names,mdata_guide.uns["guide_targets"]))

adata.uns["guide2gene"] = guide2gene

# Covariates

'''adata.obs['biological_sample']
adata.obs['guide_umi_counts']
adata.obs['n_genes_by_counts']
adata.obs['total_counts']
adata.obs['pct_counts_mt']
'''

df = pd.DataFrame({
    'biological_sample': adata.obs['biological_sample'],
    'log_guide_umi_counts': np.log1p(adata.obs['guide_umi_counts']),
    'log_n_genes_by_counts': np.log1p(adata.obs['n_genes_by_counts']),
    'log_total_counts': np.log1p(adata.obs['total_counts']),
    'log_pct_counts_mt': (adata.obs['pct_counts_mt'])
})

adata.obsm["covar"] = df


In [4]:
# add flooring for program matrix with exceesive zeros

U = adata.obsm["cnmf_usage"].copy()
U = np.maximum(U, 1e-8)
U /= U.sum(axis=1, keepdims=True)
adata.obsm["cnmf_usage"] = U

In [ ]:
# run one CRT
def run_CRT(adata, output_folder):

    # perform CRT for each condition of cells 
    #for condition in adata.obs['timepoint'].unique():
        condition = 'd3'
        adata_con = adata[adata.obs['timepoint'] == condition].copy()
        
        inputs = prepare_crt_inputs(
            adata=adata_con,
            usage_key="cnmf_usage",
            covar_key="covar",
            guide_assignment_key="guide_assignment",
            guide2gene_key="guide2gene"
        )


        out = run_all_genes_union_crt(
            inputs=inputs,
            B=5000,
            n_jobs=-1,
            calibrate_skew_normal=True,
            return_raw_pvals=True,
            return_skew_normal=True,
        )

        ntc_labels = ['non-targeting']# Identify NTC guides and build guide-frequency bins / real-gene signatures
        ntc_guides, guide_freq, guide_to_bin, real_sigs = build_ntc_group_inputs(
            inputs=inputs,
            ntc_label=ntc_labels,
            group_size=6,
            n_bins=10,
        )

        # Create multiple random partitions (ensembles) of NTC guides into 6-guide groups
        ntc_groups_ens = make_ntc_groups_ensemble(
            ntc_guides=ntc_guides,
            ntc_freq=guide_freq,
            real_gene_bin_sigs=real_sigs,
            guide_to_bin=guide_to_bin,
            n_ensemble=10,
            seed0=7,
            group_size=6,
            max_groups=None,
        )

        # Compute raw CRT p-values for each NTC group in each ensemble
        ntc_group_pvals_ens = crt_pvals_for_ntc_groups_ensemble(
            inputs=inputs,
            ntc_groups_ens=ntc_groups_ens,
            B=5000,
            seed0=23,
        )

        # Compute skew-calibrated CRT p-values for each NTC group in each ensemble
        ntc_group_pvals_skew_ens = crt_pvals_for_ntc_groups_ensemble_skew(
            inputs=inputs,
            ntc_groups_ens=ntc_groups_ens,
            B=5000,
            seed0=23,
        )

        # Build CRT-null p-values matched to NTC group units (recommended)
        guide_to_col = {g: i for i, g in enumerate(inputs.guide_names)}
        null_pvals = np.concatenate(
            [
                # Each entry is a (B, K) matrix of null p-values for one NTC group
                compute_guide_set_null_pvals(
                    guide_idx=[guide_to_col[g] for g in guides],
                    inputs=inputs,
                    B=5000,
                ).ravel()
                for groups in ntc_groups_ens
                for guides in groups.values()
            ]
        )

        ax = qq_plot_ntc_pvals(
            pvals_raw_df=out["pvals_raw_df"],
            guide2gene=adata.uns["guide2gene"],
            ntc_genes=ntc_labels,
            pvals_skew_df=out["pvals_df"],
            null_pvals=null_pvals,
            ntc_group_pvals_ens=ntc_group_pvals_ens,
            ntc_group_pvals_skew_ens=ntc_group_pvals_skew_ens,
            show_ntc_ensemble_band=True,
            show_all_pvals=True,
            title=f"QQ plot: grouped NTC controls (raw vs skew) vs CRT null for {condition}",
        )

        import matplotlib.pyplot as plt
 
        if output_folder:

            plt.savefig(f"{output_folder}/CRT_{condition}.png", dpi=100, bbox_inches='tight')
            save_result(out, output_folder, condition)

        plt.tight_layout()
        plt.show()


# save results
def save_result(out, output_folder, condition):

    pval_df = out['pvals_skew_df']
    beta_df = out['betas_df']

    pval_long = pval_df.reset_index().melt(
        id_vars='index',
        var_name='program_name',
        value_name='p-value'
    ).rename(columns={'index': 'target_name'})
    
    # Melt beta dataframe
    beta_long = beta_df.reset_index().melt(
        id_vars='index',
        var_name='program_name',
        value_name='log2FC'
    ).rename(columns={'index': 'target_name'})
    
    # Merge on program and target_gene
    result_df = pval_long.merge(
        beta_long,
        on=['program_name', 'target_name'],
        how='inner'
    )
    
    # Reorder columns
    result_df = result_df[['target_name', 'program_name',  'log2FC', 'p-value']]


    # Correct for multiple testing
    '''    
    if args.FDR_method == 'BH':

        from statsmodels.stats.multitest import multipletests
        result_df['adj_pval'] = multipletests(result_df['p-value'], method='fdr_bh')[1]

    elif args.FDR_method == 'StoreyQ':'''

    from multipy.fdr import qvalue
    import builtins
    if not hasattr(builtins, 'xrange'):
        builtins.xrange = range
    result_df['adj_pval'] = qvalue(result_df['p-value'].values, threshold=0.05, verbose=False)[1]


    #result_df.to_csv(f'{output_folder}/CRT_{condition}.txt',sep='\t',index=False)
    
    return result_df

# reformat mudata for CRT use
def reformat_data_for_CRT(mdata, mdata_guide):

    adata = mdata["cNMF"].copy()
    adata.obsm["cnmf_usage"] = np.asarray(adata.X)  # ensure dense float

    # Align gene modality to adata cells
    g = mdata_guide[adata.obs_names].copy()

    # Guide assignment (cell x guide)
    adata.obsm["guide_assignment"] = g.obsm["guide_assignment"].copy()

    # Program names
    adata.uns["program_names"] = list(adata.var_names)

    # Guide names must match columns of guide_assignment
    guide_names = list(mdata_guide.uns["guide_names"])
    adata.uns["guide_names"] = guide_names

    # guide2gene must map guide_name -> gene name (keys must match guide_names)
    '''
    guide2gene = (
        mdata_guide.uns["guide_targets"]
        .loc[guide_names]
        .to_dict()
    )
    '''

    guide2gene = dict(zip(guide_names,mdata_guide.uns["guide_targets"]))

    adata.uns["guide2gene"] = guide2gene

    # Covariates

    '''
    adata.obs['biological_sample']
    adata.obs['guide_umi_counts']
    adata.obs['n_genes_by_counts']
    adata.obs['total_counts']
    adata.obs['pct_counts_mt']
    '''

    df = pd.DataFrame({
        'biological_sample': adata.obs['biological_sample'],
        'log_guide_umi_counts': np.log1p(adata.obs['guide_umi_counts']),
        'log_n_genes_by_counts': np.log1p(adata.obs['n_genes_by_counts']),
        'log_total_counts': np.log1p(adata.obs['total_counts']),
        'log_pct_counts_mt': np.log1p(adata.obs['pct_counts_mt'])
    })

    adata.obsm["covar"] = df

    return adata


In [6]:
sel_thresh = [0.2]
components = [50]
out_dir = "/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Result"
run_name = "020326_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50"

In [ ]:
# run CRT for each K and dt
for sel_thresh in sel_thresh:
    for k in components:
        print(f"Processing K={k}, sel_thresh={sel_thresh}")

        output_folder = f"{out_dir}/{run_name}/Evaluation/{k}_{str(sel_thresh).replace('.','_')}"
        os.makedirs(output_folder, exist_ok=True)

        # Load mdata
        mdata = mu.read(f'{out_dir}/{run_name}/adata/cNMF_{k}_{str(sel_thresh).replace(".","_")}.h5mu')

        # Assign guide
        adata = reformat_data_for_CRT(mdata, mdata_guide)

        # add flooring for program matrix with exceesive zeros
        U = adata.obsm["cnmf_usage"].copy()
        U = np.maximum(U, 1e-8)
        U /= U.sum(axis=1, keepdims=True)
        adata.obsm["cnmf_usage"] = U

        # run CRT
        result_df = run_CRT(adata, output_folder)